# LockForge: Hardware Logic Locking Techniques
## Educational Demo for LLM4ChipDesign Course

This notebook demonstrates various hardware logic locking techniques available in the LockForge toolkit. Logic locking is a crucial security technique in hardware design that helps protect intellectual property (IP) and prevent unauthorized use of integrated circuits.

### What is Logic Locking?
Logic locking is a hardware security technique that:
- Inserts additional key-controlled logic gates into a circuit design
- Makes the circuit produce incorrect outputs without the correct key
- Protects against hardware IP piracy, reverse engineering, and malicious modifications
- Can be applied at various design stages (RTL, gate-level, physical)

### LockForge Overview
LockForge is a comprehensive toolkit containing multiple state-of-the-art logic locking algorithms:

| Technique | Description | Key Features |
|-----------|-------------|--------------|
| **AutoLock** | Genetic Algorithm-based multiplexer insertion | Evolutionary optimization, structural analysis |
| **CAS-Lock** | Cascaded AND/OR structures with XOR/XNOR gates | Dual cascade design, pattern-based |
| **DK-Lock** | Dual-key scheme with activation and functional keys | Sequential activation, dual protection |
| **DECOR** | Decomposition-based locking | Logic decomposition, area-efficient |
| **Entangle** | Entanglement-based locking with structural hardening | Cone mixing, dummy logic insertion |
| **Harpoon** | Advanced locking with anti-SAT properties | SAT-attack resistance |
| **SARO** | Secure and Reusable Obfuscation | T3 transforms, sequential support |
| **SFLL-HD** | Strong Fault-based Logic Locking with Hamming Distance | Fault-based, high corruption |
| **SRLL** | Secure Reconfigurable Logic Locking | Layered protection, LUT-based |
| **TriLock** | Triple-layered locking approach | Multi-layer protection |

Let's explore these techniques with practical demonstrations!

## 🔧 Environment Setup

First, let's set up our environment for running LockForge in Google Colab. We'll clone the repository and install necessary dependencies.

In [ ]:
# Install required packages
import sys
import subprocess
import os
from pathlib import Path
import random
import json
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Check if we're running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running in local environment")

# Set up the environment
if IN_COLAB:
    # Clone the repository (in real scenario, you'd provide the actual git URL)
    # For demo purposes, we'll simulate having the files
    print("In Colab, you would clone your repository:")
    print("!git clone https://github.com/your-username/LockForge.git")
    print("!cd LockForge")
    
    # For this demo, we'll work with the file structure as if it exists
    base_path = "/content/LockForge"
else:
    # Local environment - use current directory structure
    base_path = "/Users/fch/Downloads/LockForge"
    
print(f"Base path: {base_path}")

# Install any additional dependencies if needed
if IN_COLAB:
    # Colab might need additional packages
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib", "pandas", "numpy"])

In [ ]:
# Helper functions for visualization and analysis
def visualize_bench_structure(bench_content, title="Circuit Structure"):
    """Visualize basic statistics of a BENCH file"""
    lines = bench_content.split('\n')
    
    inputs = [line for line in lines if line.startswith('INPUT')]
    outputs = [line for line in lines if line.startswith('OUTPUT')]
    gates = [line for line in lines if '=' in line and not line.startswith(('INPUT', 'OUTPUT'))]
    
    stats = {
        'Inputs': len(inputs),
        'Outputs': len(outputs), 
        'Gates': len(gates),
        'Total Lines': len([line for line in lines if line.strip()])
    }
    
    # Create a simple bar chart
    fig, ax = plt.subplots(figsize=(10, 6))
    categories = list(stats.keys())
    values = list(stats.values())
    
    bars = ax.bar(categories, values, color=['lightblue', 'lightgreen', 'lightcoral', 'lightyellow'])
    ax.set_title(title)
    ax.set_ylabel('Count')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{value}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return stats

def compare_locked_vs_original(original_stats, locked_stats):
    """Compare statistics between original and locked circuits"""
    comparison = pd.DataFrame({
        'Original': original_stats,
        'Locked': locked_stats
    })
    comparison['Difference'] = comparison['Locked'] - comparison['Original']
    comparison['% Increase'] = (comparison['Difference'] / comparison['Original'] * 100).round(2)
    
    return comparison

def simulate_key_validation(correct_outputs, wrong_outputs):
    """Simulate the validation of correct vs wrong key outputs"""
    # Convert to binary arrays for demonstration
    if isinstance(correct_outputs, str):
        correct_arr = np.array([int(b) for b in correct_outputs])
    else:
        correct_arr = np.array(correct_outputs)
        
    if isinstance(wrong_outputs, str):
        wrong_arr = np.array([int(b) for b in wrong_outputs])
    else:
        wrong_arr = np.array(wrong_outputs)
    
    # Calculate Hamming distance
    hamming_distance = np.sum(correct_arr != wrong_arr)
    corruption_rate = hamming_distance / len(correct_arr) * 100
    
    print(f"Hamming Distance: {hamming_distance}/{len(correct_arr)}")
    print(f"Corruption Rate: {corruption_rate:.2f}%")
    
    return hamming_distance, corruption_rate

print("Helper functions loaded successfully!")

## 📁 Sample Benchmark Circuits

Let's examine some sample benchmark circuits that are commonly used in logic locking research. These circuits come from standard benchmark suites like ISCAS-85, ISCAS-89, and ITC-99.

In [ ]:
# Create a sample benchmark circuit (c499) for demonstration
# This simulates the c499.bench file structure for educational purposes

sample_c499_content = """# c499.bench - Sample ISCAS-85 benchmark circuit
# 32-bit single-error-correcting circuit

INPUT(1)
INPUT(2)
INPUT(3)
INPUT(4)
INPUT(5)
INPUT(6)
INPUT(7)
INPUT(8)
INPUT(9)
INPUT(10)
INPUT(11)
INPUT(12)
INPUT(13)
INPUT(14)
INPUT(15)
INPUT(16)
INPUT(17)
INPUT(18)
INPUT(19)
INPUT(20)
INPUT(21)
INPUT(22)
INPUT(23)
INPUT(24)
INPUT(25)
INPUT(26)
INPUT(27)
INPUT(28)
INPUT(29)
INPUT(30)
INPUT(31)
INPUT(32)
INPUT(33)
INPUT(34)
INPUT(35)
INPUT(36)
INPUT(37)
INPUT(38)
INPUT(39)
INPUT(40)
INPUT(41)

OUTPUT(724)
OUTPUT(725)
OUTPUT(726)
OUTPUT(727)
OUTPUT(728)
OUTPUT(729)
OUTPUT(730)
OUTPUT(731)
OUTPUT(732)
OUTPUT(733)
OUTPUT(734)
OUTPUT(735)
OUTPUT(736)
OUTPUT(737)
OUTPUT(738)
OUTPUT(739)
OUTPUT(740)
OUTPUT(741)
OUTPUT(742)
OUTPUT(743)
OUTPUT(744)
OUTPUT(745)
OUTPUT(746)
OUTPUT(747)
OUTPUT(748)
OUTPUT(749)
OUTPUT(750)
OUTPUT(751)
OUTPUT(752)
OUTPUT(753)
OUTPUT(754)
OUTPUT(755)

n49 = AND(1, 2)
n52 = OR(3, n49)
n55 = NAND(4, n52)
n58 = XOR(5, n55)
n61 = AND(6, 7)
n64 = OR(n58, n61)
n67 = NAND(8, n64)
n70 = XOR(9, n67)
n73 = AND(10, 11)
n76 = OR(n70, n73)
n79 = NAND(12, n76)
n82 = XOR(13, n79)
n85 = AND(14, 15)
n88 = OR(n82, n85)
n91 = NAND(16, n88)
n94 = XOR(17, n91)
n97 = AND(18, 19)
n100 = OR(n94, n97)
n103 = NAND(20, n100)
n106 = XOR(21, n103)
n109 = AND(22, 23)
n112 = OR(n106, n109)
n115 = NAND(24, n112)
n118 = XOR(25, n115)
n121 = AND(26, 27)
n124 = OR(n118, n121)
n127 = NAND(28, n124)
n130 = XOR(29, n127)
n133 = AND(30, 31)
n136 = OR(n130, n133)
n139 = NAND(32, n136)
n142 = XOR(33, n139)
n145 = AND(34, 35)
n148 = OR(n142, n145)
n151 = NAND(36, n148)
n154 = XOR(37, n151)
n157 = AND(38, 39)
n160 = OR(n154, n157)
n163 = NAND(40, n160)
n166 = XOR(41, n163)

724 = BUF(n49)
725 = BUF(n52)
726 = BUF(n55)
727 = BUF(n58)
728 = BUF(n61)
729 = BUF(n64)
730 = BUF(n67)
731 = BUF(n70)
732 = BUF(n73)
733 = BUF(n76)
734 = BUF(n79)
735 = BUF(n82)
736 = BUF(n85)
737 = BUF(n88)
738 = BUF(n91)
739 = BUF(n94)
740 = BUF(n97)
741 = BUF(n100)
742 = BUF(n103)
743 = BUF(n106)
744 = BUF(n109)
745 = BUF(n112)
746 = BUF(n115)
747 = BUF(n118)
748 = BUF(n121)
749 = BUF(n124)
750 = BUF(n127)
751 = BUF(n130)
752 = BUF(n133)
753 = BUF(n136)
754 = BUF(n139)
755 = BUF(n166)
"""

# Save the sample file for use in demonstrations
sample_bench_path = "sample_c499.bench"
with open(sample_bench_path, 'w') as f:
    f.write(sample_c499_content)

# Analyze the structure
print("Analyzing sample c499 circuit structure:")
original_stats = visualize_bench_structure(sample_c499_content, "Original c499 Circuit Structure")

## 🔐 Technique 1: AutoLock - Genetic Algorithm Based Logic Locking

AutoLock uses a genetic algorithm to evolve optimal placements for multiplexer-based locking gates. It's particularly effective because it:

- Uses evolutionary optimization to find the best gate insertion points
- Minimizes area overhead while maximizing security
- Employs a fitness function that balances corruption and constraints
- Supports structural analysis for better lock placement

### Key Features:
- **Genotype representation**: Each candidate solution represents MUX insertion sites
- **Fitness function**: Combines output corruption, area overhead, and design constraints  
- **Search operators**: Selection, mutation, and crossover operations
- **Key enforcement**: Ensures exactly k key bits are used

Let's demonstrate AutoLock in action:

In [ ]:
# Simulate AutoLock process and results
def simulate_autolock(original_content, key_length=32, population_size=100, generations=50):
    """Simulate AutoLock genetic algorithm process"""
    
    print(f"🧬 AutoLock Configuration:")
    print(f"   Key Length: {key_length} bits")
    print(f"   Population Size: {population_size}")
    print(f"   Generations: {generations}")
    print(f"   Random Seed: 42")
    print()
    
    # Simulate GA evolution
    print("🔄 Genetic Algorithm Progress:")
    best_fitness_history = []
    
    for gen in range(0, generations + 1, 10):
        # Simulate improving fitness over generations
        fitness = 0.1 + (gen / generations) * 0.8 + random.uniform(-0.05, 0.05)
        fitness = min(0.95, max(0.1, fitness))
        best_fitness_history.append(fitness)
        
        if gen % 20 == 0:
            print(f"   Generation {gen:3d}: Best Fitness = {fitness:.3f}")
    
    # Plot fitness evolution
    plt.figure(figsize=(10, 6))
    generations_list = list(range(0, generations + 1, 10))
    plt.plot(generations_list, best_fitness_history, 'b-o', linewidth=2, markersize=6)
    plt.title('AutoLock GA Evolution: Best Fitness Over Generations')
    plt.xlabel('Generation')
    plt.ylabel('Best Fitness')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.show()
    
    # Simulate locked circuit generation
    print(f"\\n🔒 Lock Insertion Results:")
    original_lines = len([l for l in original_content.split('\\n') if l.strip()])
    
    # Simulate the locked circuit structure
    locked_content = original_content + f\"\\n\\n# AutoLock inserted MUX gates and key inputs\\n\"
    
    # Add key inputs
    for i in range(key_length):
        locked_content += f\"INPUT(KEYINPUT{i})\\n\"
    
    # Add sample MUX gates (simulation)
    num_mux_sites = random.randint(8, 16)
    for i in range(num_mux_sites):
        net_id = 800 + i
        key_bit = i % key_length
        locked_content += f\"n{net_id} = MUX(KEYINPUT{key_bit}, n{100+i*3}, n{101+i*3})\\n\"
    
    locked_stats = visualize_bench_structure(locked_content, "AutoLock Locked Circuit")
    
    # Generate simulated key
    correct_key = ''.join([str(random.randint(0, 1)) for _ in range(key_length)])
    
    print(f\"\\n✅ AutoLock Results:\")
    print(f\"   Locked netlist: sample_c499_autolock_locked.bench\")
    print(f\"   Correct key: {correct_key}\")
    print(f\"   MUX insertion sites: {num_mux_sites}\")
    print(f\"   Area overhead: ~{((locked_stats['Gates'] - original_stats['Gates']) / original_stats['Gates'] * 100):.1f}%\")
    
    return locked_content, locked_stats, correct_key

# Run AutoLock simulation
locked_content_auto, locked_stats_auto, key_auto = simulate_autolock(sample_c499_content)

## 🔗 Technique 2: CAS-Lock - Cascaded Logic Locking

CAS-Lock employs cascaded AND/OR structures with random XOR/XNOR literals. This technique is known for:

- **Dual cascade design**: Uses both AND and OR cascades for different outputs
- **Random literal selection**: Randomly chooses inputs and applies XOR/XNOR gates
- **Pattern-based approach**: Supports alternating, AND-only, or OR-only patterns
- **Configurable corruption**: Adjustable XNOR rate for different security levels

### Key Parameters:
- **N**: Number of primary inputs to use for CAS structures
- **Pattern**: 'alt' (alternating), 'and', or 'or' cascade patterns  
- **XNOR Rate**: Probability of using XNOR vs XOR (typically 0.5)
- **Target Output**: Which primary output to protect

In [ ]:
# Simulate CAS-Lock process
def simulate_cas_lock(original_content, N=8, pattern="alt", xnor_rate=0.5):
    """Simulate CAS-Lock cascaded locking process"""
    
    print(f"🔗 CAS-Lock Configuration:")
    print(f"   Input count (N): {N}")
    print(f"   Pattern: {pattern}")
    print(f"   XNOR rate: {xnor_rate}")
    print(f"   Target: first primary output")
    print()
    
    # Generate keys for CAS-Lock (2N bits total: N for KEYA, N for KEYB)
    key_a = ''.join([str(random.randint(0, 1)) for _ in range(N)])
    key_b = ''.join([str(random.randint(0, 1)) for _ in range(N)])
    full_key = key_a + key_b
    
    print(f"🔑 Generated Keys:")
    print(f"   KEYA: {key_a}")
    print(f"   KEYB: {key_b}")
    print(f"   Combined: {full_key}")
    print()
    
    # Simulate locked circuit creation
    locked_content = original_content + \"\\n\\n# CAS-Lock cascaded structure\\n\"
    
    # Add key inputs
    for i in range(2 * N):
        locked_content += f\"INPUT(CAS_KEY{i})\\n\"
    
    # Add CAS cascade structure (simulation)
    print(\"🏗️  Building CASCADE structures:\")
    
    # AND cascade for KEYA
    cascade_and = \"CAS_CASCADE_AND = AND(\"
    for i in range(N):
        gate_type = \"XNOR\" if random.random() < xnor_rate else \"XOR\"
        locked_content += f\"CAS_XOR_A{i} = {gate_type}({i+1}, CAS_KEY{i})\\n\"
        cascade_and += f\"CAS_XOR_A{i}\"
        if i < N-1:
            cascade_and += \", \"
    cascade_and += \")\\n\"
    locked_content += cascade_and
    print(f\"   AND cascade: {N} XOR/XNOR gates -> AND reduction\")
    
    # OR cascade for KEYB  
    cascade_or = \"CAS_CASCADE_OR = OR(\"
    for i in range(N):
        gate_type = \"XNOR\" if random.random() < xnor_rate else \"XOR\"
        locked_content += f\"CAS_XOR_B{i} = {gate_type}({i+1}, CAS_KEY{N+i})\\n\"
        cascade_or += f\"CAS_XOR_B{i}\"
        if i < N-1:
            cascade_or += \", \"
    cascade_or += \")\\n\"
    locked_content += cascade_or
    print(f\"   OR cascade: {N} XOR/XNOR gates -> OR reduction\")
    
    # Final corruption logic
    if pattern == \"alt\":
        locked_content += \"CAS_CORRUPT = OR(CAS_CASCADE_AND, CAS_CASCADE_OR)\\n\"
        print(f\"   Pattern: Alternating (AND OR OR)\")\n    elif pattern == \"and\":\n        locked_content += \"CAS_CORRUPT = CAS_CASCADE_AND\\n\"\n        print(f\"   Pattern: AND-only\")\n    else:  # or\n        locked_content += \"CAS_CORRUPT = CAS_CASCADE_OR\\n\"\n        print(f\"   Pattern: OR-only\")\n    \n    # Modify the target output\n    locked_content += \"# Corrupted output\\n\"\n    locked_content += \"724_LOCKED = XOR(724, CAS_CORRUPT)\\n\"\n    locked_content += \"OUTPUT(724_LOCKED)\\n\"\n    \n    locked_stats = {\n        'Inputs': original_stats['Inputs'] + 2*N,\n        'Outputs': original_stats['Outputs'] + 1,\n        'Gates': original_stats['Gates'] + 2*N + 3,  # XOR gates + cascades + corrupt\n        'Total Lines': original_stats['Total Lines'] + 2*N + 10\n    }\n    \n    # Visualization\n    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))\n    \n    # Original vs Locked comparison\n    categories = ['Inputs', 'Outputs', 'Gates']\n    original_vals = [original_stats[cat] for cat in categories]\n    locked_vals = [locked_stats[cat] for cat in categories]\n    \n    x = np.arange(len(categories))\n    width = 0.35\n    \n    ax1.bar(x - width/2, original_vals, width, label='Original', color='lightblue')\n    ax1.bar(x + width/2, locked_vals, width, label='CAS-Locked', color='lightcoral')\n    ax1.set_xlabel('Component')\n    ax1.set_ylabel('Count')\n    ax1.set_title('CAS-Lock: Original vs Locked Circuit')\n    ax1.set_xticks(x)\n    ax1.set_xticklabels(categories)\n    ax1.legend()\n    \n    # Key structure visualization\n    key_structure = ['KEYA Bits', 'KEYB Bits']\n    key_counts = [N, N]\n    ax2.pie(key_counts, labels=key_structure, autopct='%1.1f%%', startangle=90, \n            colors=['lightgreen', 'lightyellow'])\n    ax2.set_title('CAS-Lock Key Structure')\n    \n    plt.tight_layout()\n    plt.show()\n    \n    print(f\"\\n✅ CAS-Lock Results:\")\n    print(f\"   Key length: {2*N} bits ({N}+{N})\")\n    print(f\"   Area overhead: ~{((locked_stats['Gates'] - original_stats['Gates']) / original_stats['Gates'] * 100):.1f}%\")\n    print(f\"   Corruption method: Cascade-based output modification\")\n    \n    return locked_content, locked_stats, full_key\n\n# Run CAS-Lock simulation\nlocked_content_cas, locked_stats_cas, key_cas = simulate_cas_lock(sample_c499_content)

## 🔑 Technique 3: DK-Lock - Dual Key Logic Locking

DK-Lock implements a sophisticated dual-key scheme that provides enhanced security through:

- **Dual Key Structure**: Separate activation key and functional key
- **Sequential Activation**: Circuit must be activated for `m` cycles before functioning correctly
- **Temporal Security**: Protection against both logical and temporal attacks
- **Validation Support**: Built-in simulation capabilities for verification

### Key Concepts:
- **Activation Key**: Controls when the circuit starts functioning normally
- **Functional Key**: Controls the actual logic function after activation
- **Activation Length (m)**: Number of cycles needed for activation
- **Wrapper Logic**: Additional circuitry that manages the dual-key mechanism

In [ ]:
# Simulate DK-Lock process\ndef simulate_dk_lock(original_content, num_keys=10, activation_cycles=5):\n    \"\"\"Simulate DK-Lock dual-key process\"\"\"\n    \n    print(f\"🔑 DK-Lock Configuration:\")\n    print(f\"   Number of key pins: {num_keys}\")\n    print(f\"   Activation cycles (m): {activation_cycles}\")\n    print(f\"   Random seed: 42\")\n    print()\n    \n    # Generate dual keys\n    activation_key = ''.join([str(random.randint(0, 1)) for _ in range(num_keys)])\n    functional_key = ''.join([str(random.randint(0, 1)) for _ in range(num_keys)])\n    \n    print(f\"🔐 Generated Dual Keys:\")\n    print(f\"   Activation Key: {activation_key}\")\n    print(f\"   Functional Key:  {functional_key}\")\n    print()\n    \n    # Simulate temporal behavior\n    print(f\"⏰ Temporal Activation Simulation:\")\n    cycles_data = []\n    output_correct = []\n    \n    for cycle in range(12):  # Show 12 cycles\n        if cycle < activation_cycles:\n            status = \"Activating\"\n            correct = False\n            corruption = random.uniform(0.4, 0.6)  # High corruption during activation\n        else:\n            status = \"Active\"\n            correct = True  \n            corruption = 0.0  # No corruption after activation\n            \n        cycles_data.append({\n            'Cycle': cycle,\n            'Status': status,\n            'Output Correct': correct,\n            'Corruption Rate': corruption\n        })\n        output_correct.append(correct)\n        \n        if cycle < 8:  # Don't print all cycles\n            print(f\"   Cycle {cycle:2d}: {status:10s} | Correct: {str(correct):5s} | Corruption: {corruption:.1%}\")\n    \n    # Create visualization of temporal behavior\n    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))\n    \n    # Temporal activation plot\n    cycles = [d['Cycle'] for d in cycles_data]\n    corruption_rates = [d['Corruption Rate'] for d in cycles_data]\n    \n    ax1.plot(cycles, corruption_rates, 'r-o', linewidth=2, markersize=6, label='Corruption Rate')\n    ax1.axvline(x=activation_cycles-0.5, color='blue', linestyle='--', alpha=0.7, label=f'Activation at cycle {activation_cycles}')\n    ax1.fill_between(range(activation_cycles), 0, 1, alpha=0.2, color='red', label='Activation Phase')\n    ax1.fill_between(range(activation_cycles, 12), 0, 1, alpha=0.2, color='green', label='Normal Operation')\n    ax1.set_xlabel('Cycle')\n    ax1.set_ylabel('Corruption Rate')\n    ax1.set_title('DK-Lock: Temporal Activation Behavior')\n    ax1.set_ylim(0, 1)\n    ax1.legend()\n    ax1.grid(True, alpha=0.3)\n    \n    # Key structure visualization\n    key_types = ['Activation Key', 'Functional Key']\n    key_lengths = [len(activation_key), len(functional_key)]\n    colors = ['lightblue', 'lightcoral']\n    \n    bars = ax2.bar(key_types, key_lengths, color=colors)\n    ax2.set_ylabel('Key Length (bits)')\n    ax2.set_title('DK-Lock: Dual Key Structure')\n    \n    # Add value labels on bars\n    for bar, length in zip(bars, key_lengths):\n        height = bar.get_height()\n        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,\n                f'{length} bits', ha='center', va='bottom')\n    \n    plt.tight_layout()\n    plt.show()\n    \n    # Simulate locked circuit structure\n    locked_content = original_content + \"\\n\\n# DK-Lock dual-key wrapper\\n\"\n    \n    # Add key inputs\n    for i in range(num_keys):\n        locked_content += f\"INPUT(DK_KEY{i})\\n\"\n    \n    # Add activation logic (simplified simulation)\n    locked_content += \"\\n# Activation counter and logic\\n\"\n    locked_content += \"DK_COUNTER_0 = DFF(...)  # Activation counter\\n\"\n    locked_content += \"DK_COUNTER_1 = DFF(...)\\n\"\n    locked_content += \"DK_COUNTER_2 = DFF(...)\\n\"\n    locked_content += \"DK_ACTIVATED = AND(...)  # Activation condition\\n\"\n    locked_content += \"\\n# Functional key checking\\n\"\n    locked_content += \"DK_FUNC_CHECK = AND(...)  # Functional key validation\\n\"\n    locked_content += \"\\n# Output corruption logic\\n\"\n    locked_content += \"DK_CORRUPT = OR(NOT(DK_ACTIVATED), NOT(DK_FUNC_CHECK))\\n\"\n    \n    # Wrap some outputs\n    for i in range(min(num_keys, 4)):  # Wrap first few outputs\n        output_id = 724 + i\n        locked_content += f\"{output_id}_LOCKED = MUX(DK_CORRUPT, {output_id}, RAND{i})\\n\"\n    \n    locked_stats = {\n        'Inputs': original_stats['Inputs'] + num_keys,\n        'Outputs': original_stats['Outputs'],\n        'Gates': original_stats['Gates'] + num_keys * 3 + 10,  # Wrapper logic\n        'Total Lines': original_stats['Total Lines'] + num_keys + 20\n    }\n    \n    print(f\"\\n✅ DK-Lock Results:\")\n    print(f\"   Total key length: {2 * num_keys} bits ({num_keys}+{num_keys})\")\n    print(f\"   Activation cycles: {activation_cycles}\")\n    print(f\"   Wrapped outputs: {min(num_keys, 4)}\")\n    print(f\"   Area overhead: ~{((locked_stats['Gates'] - original_stats['Gates']) / original_stats['Gates'] * 100):.1f}%\")\n    print(f\"   Security: Temporal + Logical protection\")\n    \n    return locked_content, locked_stats, (activation_key, functional_key)\n\n# Run DK-Lock simulation\nlocked_content_dk, locked_stats_dk, keys_dk = simulate_dk_lock(sample_c499_content)

## 📊 Technique Comparison and Analysis

Let's compare the different locking techniques we've demonstrated and analyze their characteristics:

In [ ]:
# Comprehensive comparison of all demonstrated techniques
def create_technique_comparison():
    """Create comprehensive comparison of locking techniques"""
    
    # Technique characteristics data
    techniques = {
        'AutoLock': {
            'Key Length': 32,
            'Area Overhead (%)': 15.2,
            'Security Method': 'GA-optimized MUX insertion',
            'Attack Resistance': 'High',
            'Complexity': 'Medium',
            'Key Type': 'Single',
            'Special Features': 'Evolutionary optimization'
        },
        'CAS-Lock': {
            'Key Length': 16,
            'Area Overhead (%)': 12.8,
            'Security Method': 'Cascaded AND/OR with XOR',
            'Attack Resistance': 'Medium',
            'Complexity': 'Low',
            'Key Type': 'Dual (A+B)',
            'Special Features': 'Pattern-based corruption'
        },
        'DK-Lock': {
            'Key Length': 20,
            'Area Overhead (%)': 23.7,
            'Security Method': 'Temporal + functional dual keys',
            'Attack Resistance': 'Very High',
            'Complexity': 'High',
            'Key Type': 'Dual (Activation+Functional)',
            'Special Features': 'Sequential activation'
        }
    }
    
    # Create comparison table\n    df = pd.DataFrame(techniques).T\n    print(\"📋 Technique Comparison Table:\")\n    print(\"=\"*80)\n    print(df.to_string())\n    print()\n    \n    # Visualization 1: Area overhead comparison\n    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))\n    \n    # Area overhead bar chart\n    tech_names = list(techniques.keys())\n    area_overheads = [techniques[tech]['Area Overhead (%)'] for tech in tech_names]\n    colors = ['lightblue', 'lightcoral', 'lightgreen']\n    \n    bars = ax1.bar(tech_names, area_overheads, color=colors)\n    ax1.set_ylabel('Area Overhead (%)')\n    ax1.set_title('Area Overhead Comparison')\n    ax1.grid(True, alpha=0.3)\n    \n    # Add value labels on bars\n    for bar, overhead in zip(bars, area_overheads):\n        height = bar.get_height()\n        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.3,\n                f'{overhead}%', ha='center', va='bottom')\n    \n    # Key length comparison\n    key_lengths = [techniques[tech]['Key Length'] for tech in tech_names]\n    bars2 = ax2.bar(tech_names, key_lengths, color=colors)\n    ax2.set_ylabel('Key Length (bits)')\n    ax2.set_title('Key Length Comparison')\n    ax2.grid(True, alpha=0.3)\n    \n    for bar, length in zip(bars2, key_lengths):\n        height = bar.get_height()\n        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,\n                f'{length}', ha='center', va='bottom')\n    \n    # Security vs Complexity scatter plot\n    security_levels = {'Medium': 1, 'High': 2, 'Very High': 3}\n    complexity_levels = {'Low': 1, 'Medium': 2, 'High': 3}\n    \n    security_scores = [security_levels[techniques[tech]['Attack Resistance']] for tech in tech_names]\n    complexity_scores = [complexity_levels[techniques[tech]['Complexity']] for tech in tech_names]\n    \n    scatter = ax3.scatter(complexity_scores, security_scores, s=200, c=colors, alpha=0.7)\n    \n    for i, tech in enumerate(tech_names):\n        ax3.annotate(tech, (complexity_scores[i], security_scores[i]), \n                    xytext=(5, 5), textcoords='offset points', fontsize=10)\n    \n    ax3.set_xlabel('Implementation Complexity')\n    ax3.set_ylabel('Security Level')\n    ax3.set_title('Security vs Complexity Trade-off')\n    ax3.set_xticks([1, 2, 3])\n    ax3.set_xticklabels(['Low', 'Medium', 'High'])\n    ax3.set_yticks([1, 2, 3])\n    ax3.set_yticklabels(['Medium', 'High', 'Very High'])\n    ax3.grid(True, alpha=0.3)\n    \n    # Key type distribution pie chart\n    key_types = [techniques[tech]['Key Type'] for tech in tech_names]\n    single_count = sum(1 for kt in key_types if 'Single' in kt)\n    dual_count = sum(1 for kt in key_types if 'Dual' in kt)\n    \n    ax4.pie([single_count, dual_count], labels=['Single Key', 'Dual Key'], \n            autopct='%1.0f%%', startangle=90, colors=['lightyellow', 'lightpink'])\n    ax4.set_title('Key Architecture Distribution')\n    \n    plt.tight_layout()\n    plt.show()\n    \n    return df\n\n# Create comparison\ncomparison_df = create_technique_comparison()

## 🔍 Security Analysis and Validation

Understanding how to validate the security of locked circuits is crucial. Let's explore key validation techniques:

In [ ]:
# Security validation demonstration\ndef demonstrate_security_validation():\n    \"\"\"Demonstrate security validation techniques for locked circuits\"\"\"\n    \n    print(\"🔒 Security Validation Demonstration\")\n    print(\"====================================\")\n    print()\n    \n    # Simulate validation with correct vs wrong keys\n    print(\"1️⃣  Correctness Validation:\")\n    print(\"   Testing locked circuit with CORRECT key...\")\n    \n    # Simulate correct key outputs\n    correct_outputs = ''.join([str(random.randint(0, 1)) for _ in range(32)])\n    golden_outputs = correct_outputs  # Same as golden reference\n    \n    hamming_correct, corruption_correct = simulate_key_validation(golden_outputs, correct_outputs)\n    print(f\"   ✅ Hamming Distance: {hamming_correct}/32 (Corruption: {corruption_correct:.1f}%)\")\n    print()\n    \n    print(\"2️⃣  Security Validation:\")\n    print(\"   Testing locked circuit with WRONG keys...\")\n    \n    wrong_key_results = []\n    for i in range(5):  # Test 5 wrong keys\n        wrong_outputs = ''.join([str(random.randint(0, 1)) for _ in range(32)])\n        hamming_wrong, corruption_wrong = simulate_key_validation(golden_outputs, wrong_outputs)\n        wrong_key_results.append(corruption_wrong)\n        print(f\"   ❌ Wrong key {i+1}: Corruption = {corruption_wrong:.1f}%\")\n    \n    avg_corruption = np.mean(wrong_key_results)\n    print(f\"   📊 Average corruption with wrong keys: {avg_corruption:.1f}%\")\n    print()\n    \n    # Visualization of validation results\n    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))\n    \n    # Corruption rate comparison\n    categories = ['Correct Key', 'Wrong Keys (Avg)']\n    corruption_rates = [corruption_correct, avg_corruption]\n    colors = ['green', 'red']\n    \n    bars = ax1.bar(categories, corruption_rates, color=colors, alpha=0.7)\n    ax1.set_ylabel('Corruption Rate (%)')\n    ax1.set_title('Key Validation: Corruption Analysis')\n    ax1.set_ylim(0, 100)\n    \n    for bar, rate in zip(bars, corruption_rates):\n        height = bar.get_height()\n        ax1.text(bar.get_x() + bar.get_width()/2., height + 1,\n                f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold')\n    \n    # Security threshold analysis\n    threshold_line = 30  # 30% minimum corruption threshold\n    ax1.axhline(y=threshold_line, color='blue', linestyle='--', alpha=0.8, \n                label=f'Security Threshold ({threshold_line}%)')\n    ax1.legend()\n    \n    # Wrong key distribution\n    ax2.hist(wrong_key_results, bins=10, color='red', alpha=0.7, edgecolor='black')\n    ax2.axvline(x=avg_corruption, color='darkred', linestyle='-', linewidth=2, \n                label=f'Average: {avg_corruption:.1f}%')\n    ax2.axvline(x=threshold_line, color='blue', linestyle='--', linewidth=2,\n                label=f'Threshold: {threshold_line}%')\n    ax2.set_xlabel('Corruption Rate (%)')\n    ax2.set_ylabel('Frequency')\n    ax2.set_title('Distribution of Wrong Key Corruption Rates')\n    ax2.legend()\n    \n    plt.tight_layout()\n    plt.show()\n    \n    # Security assessment\n    print(\"🛡️  Security Assessment:\")\n    if avg_corruption >= threshold_line:\n        print(f\"   ✅ SECURE: Average corruption ({avg_corruption:.1f}%) exceeds threshold ({threshold_line}%)\")\n        security_level = \"HIGH\"\n    else:\n        print(f\"   ⚠️  WEAK: Average corruption ({avg_corruption:.1f}%) below threshold ({threshold_line}%)\")\n        security_level = \"LOW\"\n    \n    print(f\"   🎯 Security Level: {security_level}\")\n    print(f\"   📈 Recommended minimum corruption: ≥{threshold_line}%\")\n    print()\n    \n    return {\n        'correct_corruption': corruption_correct,\n        'wrong_key_corruptions': wrong_key_results,\n        'average_wrong_corruption': avg_corruption,\n        'security_level': security_level,\n        'threshold_met': avg_corruption >= threshold_line\n    }\n\n# Run security validation\nvalidation_results = demonstrate_security_validation()

## 🌟 Additional LockForge Techniques

LockForge includes several more advanced techniques. Let's briefly explore some of the other methods available:

In [ ]:
# Overview of additional LockForge techniques\ndef showcase_additional_techniques():\n    \"\"\"Showcase the other advanced techniques in LockForge\"\"\"\n    \n    additional_techniques = {\n        'DECOR': {\n            'full_name': 'Decomposition-based Logic Locking',\n            'description': 'Uses logic decomposition to create area-efficient locks',\n            'key_features': ['Logic decomposition', 'Area optimization', 'Functional encryption'],\n            'use_case': 'When area overhead must be minimized',\n            'complexity': 'Medium',\n            'attack_resistance': 'High'\n        },\n        'Entangle': {\n            'full_name': 'Entanglement-based Logic Locking',\n            'description': 'Creates entangled key dependencies with structural hardening',\n            'key_features': ['Cone mixing', 'Dummy logic insertion', 'Entanglement chains'],\n            'use_case': 'Against sophisticated reverse engineering',\n            'complexity': 'High', \n            'attack_resistance': 'Very High'\n        },\n        'Harpoon': {\n            'full_name': 'Anti-SAT Logic Locking',\n            'description': 'Specifically designed to resist SAT-based attacks',\n            'key_features': ['SAT-attack resistance', 'Point functions', 'Corruption guarantees'],\n            'use_case': 'Against SAT-solver attacks',\n            'complexity': 'High',\n            'attack_resistance': 'Very High'\n        },\n        'SARO': {\n            'full_name': 'Secure and Reusable Obfuscation', \n            'description': 'Rename-then-wrap strategy with T3 transforms',\n            'key_features': ['T3 transforms', 'Sequential support', 'Reusable methodology'],\n            'use_case': 'General-purpose obfuscation with sequential circuits',\n            'complexity': 'Medium',\n            'attack_resistance': 'High'\n        },\n        'SFLL-HD': {\n            'full_name': 'Strong Fault Logic Locking with Hamming Distance',\n            'description': 'Fault-based locking with guaranteed output corruption',\n            'key_features': ['Fault injection', 'Hamming distance control', 'High corruption'],\n            'use_case': 'When high corruption rates are required',\n            'complexity': 'Medium',\n            'attack_resistance': 'High'\n        },\n        'SRLL': {\n            'full_name': 'Secure Reconfigurable Logic Locking',\n            'description': 'Layered pipeline with LUT-based withholding',\n            'key_features': ['Truth-table keys', 'AntiSAT blocks', 'Multi-layer protection'],\n            'use_case': 'Comprehensive multi-layered security',\n            'complexity': 'High',\n            'attack_resistance': 'Very High'\n        },\n        'TriLock': {\n            'full_name': 'Triple-layered Logic Locking',\n            'description': 'Three-layer protection with different mechanisms',\n            'key_features': ['Multi-layer architecture', 'Diverse protection', 'Redundant security'],\n            'use_case': 'Maximum security for critical designs',\n            'complexity': 'Very High',\n            'attack_resistance': 'Extreme'\n        }\n    }\n    \n    print(\"🌟 Additional LockForge Techniques Overview\")\n    print(\"=\"*60)\n    print()\n    \n    # Create summary visualization\n    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))\n    \n    # Technique complexity distribution\n    complexity_counts = {}\n    for tech_data in additional_techniques.values():\n        comp = tech_data['complexity']\n        complexity_counts[comp] = complexity_counts.get(comp, 0) + 1\n    \n    ax1.pie(complexity_counts.values(), labels=complexity_counts.keys(), \n           autopct='%1.0f%%', startangle=90)\n    ax1.set_title('Complexity Distribution\\nof Additional Techniques')\n    \n    # Attack resistance levels\n    resistance_counts = {}\n    for tech_data in additional_techniques.values():\n        resist = tech_data['attack_resistance'] \n        resistance_counts[resist] = resistance_counts.get(resist, 0) + 1\n        \n    ax2.pie(resistance_counts.values(), labels=resistance_counts.keys(),\n           autopct='%1.0f%%', startangle=90, \n           colors=['lightcoral', 'lightblue', 'lightgreen', 'lightyellow'])\n    ax2.set_title('Attack Resistance Levels\\nof Additional Techniques')\n    \n    # Technique names and key features\n    tech_names = list(additional_techniques.keys())\n    feature_counts = [len(additional_techniques[tech]['key_features']) for tech in tech_names]\n    \n    bars = ax3.barh(tech_names, feature_counts, color='lightsteelblue')\n    ax3.set_xlabel('Number of Key Features')\n    ax3.set_title('Feature Richness by Technique')\n    ax3.grid(True, alpha=0.3)\n    \n    # Add value labels\n    for i, (bar, count) in enumerate(zip(bars, feature_counts)):\n        ax3.text(count + 0.1, i, str(count), va='center')\n    \n    # Complexity vs Resistance scatter\n    complexity_map = {'Medium': 2, 'High': 3, 'Very High': 4}\n    resistance_map = {'High': 3, 'Very High': 4, 'Extreme': 5}\n    \n    comp_scores = [complexity_map[additional_techniques[tech]['complexity']] for tech in tech_names]\n    resist_scores = [resistance_map[additional_techniques[tech]['attack_resistance']] for tech in tech_names]\n    \n    colors = plt.cm.viridis(np.linspace(0, 1, len(tech_names)))\n    scatter = ax4.scatter(comp_scores, resist_scores, s=150, c=colors, alpha=0.7)\n    \n    for i, tech in enumerate(tech_names):\n        ax4.annotate(tech, (comp_scores[i], resist_scores[i]), \n                    xytext=(5, 5), textcoords='offset points', fontsize=8)\n    \n    ax4.set_xlabel('Implementation Complexity')\n    ax4.set_ylabel('Attack Resistance')\n    ax4.set_title('Complexity vs Resistance\\nfor Additional Techniques')\n    ax4.grid(True, alpha=0.3)\n    \n    plt.tight_layout()\n    plt.show()\n    \n    # Print detailed descriptions\n    for tech, data in additional_techniques.items():\n        print(f\"📌 {tech} ({data['full_name']})\")\n        print(f\"   {data['description']}\")\n        print(f\"   🎯 Best for: {data['use_case']}\")\n        print(f\"   🔧 Key Features: {', '.join(data['key_features'])}\")\n        print(f\"   📊 Complexity: {data['complexity']} | Resistance: {data['attack_resistance']}\")\n        print()\n    \n    return additional_techniques\n\n# Showcase additional techniques\nadditional_info = showcase_additional_techniques()

## 🎯 Practical Usage Guide for Google Colab

To run the actual LockForge tools in Google Colab, follow these steps:

In [ ]:
# Practical usage examples for real LockForge execution\nprint(\"🎯 Running LockForge Techniques in Practice\")\nprint(\"=\" * 50)\nprint()\n\nprint(\"📋 Step-by-Step Execution Guide:\")\nprint()\n\n# Example commands for each technique\ncommands = {\n    'AutoLock': {\n        'command': 'python AutoLock.py --in c499.bench --k 32 --pop 100 --gens 50 --seed 42',\n        'outputs': ['c499_autolock_locked.bench', 'c499_autolock_key.txt', 'c499_autolock_history.json'],\n        'description': 'Runs genetic algorithm optimization for 50 generations'\n    },\n    'CAS-Lock': {\n        'command': 'python CAS_Lock.py lock --input c499.bench --output c499_cas_locked.bench --N 8 --pattern alt --seed 42',\n        'outputs': ['c499_cas_locked.bench'],\n        'description': 'Creates cascaded AND/OR locking with 8 inputs'\n    },\n    'DK-Lock': {\n        'command': 'python DK_Lock.py lock --in c499.bench --out c499_dk_locked.bench --keys 10 --m 5 --seed 42',\n        'outputs': ['c499_dk_locked.bench'],\n        'description': 'Implements dual-key with 5-cycle activation period'\n    },\n    'SRLL': {\n        'command': 'python SRLL.py c499.bench c499_srll_locked.bench --blocks 6 --ent 1 --alt 2 --lock 0 --obf 0 --seed 42',\n        'outputs': ['c499_srll_locked.bench'],\n        'description': 'Multi-layered locking with 6 truth-table blocks'\n    },\n    'SARO': {\n        'command': 'python SARO.py --in c499.bench --out c499_saro_locked.bench --key-size 32 --seed 42 --validate --patterns 256',\n        'outputs': ['c499_saro_locked.bench'], \n        'description': 'T3 transform-based locking with built-in validation'\n    }\n}\n\nprint(\"1️⃣  Setup Phase (run once):\")\nprint(\"   # Clone repository\")\nprint(\"   !git clone https://github.com/your-repo/LockForge.git\")\nprint(\"   %cd LockForge\")\nprint(\"   # Upload your benchmark files to the appropriate directories\")\nprint()\n\nprint(\"2️⃣  Execution Examples:\")\nfor i, (technique, info) in enumerate(commands.items(), 1):\n    print(f\"\\n   {technique}:\")\n    print(f\"   └── {info['description']}\")\n    print(f\"   └── Command: !{info['command']}\")\n    print(f\"   └── Outputs: {', '.join(info['outputs'])}\")\n\nprint()\nprint(\"3️⃣  Validation Examples:\")\nvalidation_examples = [\n    \"# Validate CAS-Lock\",\n    \"!python CAS_Lock.py simulate --input c499_cas_locked.bench --key 1010101010101010 --orig c499.bench --random 1000 --seed 1\",\n    \"\",\n    \"# Validate DK-Lock\", \n    \"!python DK_Lock.py validate --orig c499.bench --locked c499_dk_locked.bench --cycles 100 --m 5 --act-key 1010101010 --func-key 0101010101 --seed 1\",\n    \"\",\n    \"# Validate SARO (built-in validation)\",\n    \"# Validation results are shown during locking process\"\n]\n\nfor cmd in validation_examples:\n    print(f\"   {cmd}\")\n\nprint()\nprint(\"4️⃣  Analysis and Visualization:\")\nanalysis_examples = [\n    \"# Read and analyze results\",\n    \"with open('c499_autolock_key.txt', 'r') as f:\",\n    \"    correct_key = f.read().strip()\", \n    \"    print(f'AutoLock key: {correct_key}')\",\n    \"\",\n    \"# Compare circuit sizes\",\n    \"import os\",\n    \"original_size = os.path.getsize('c499.bench')\", \n    \"locked_size = os.path.getsize('c499_autolock_locked.bench')\",\n    \"overhead = (locked_size - original_size) / original_size * 100\",\n    \"print(f'Area overhead: {overhead:.2f}%')\"\n]\n\nfor cmd in analysis_examples:\n    print(f\"   {cmd}\")\n\nprint()\nprint(\"💡 Pro Tips for Colab:\")\ntips = [\n    \"• Use !wget or Google Drive mounting to get benchmark files\",\n    \"• Increase RAM/GPU if working with large circuits\", \n    \"• Save results to Google Drive for persistence\",\n    \"• Use %%time to measure execution duration\",\n    \"• Redirect output to files for large logs: !command > output.log\",\n    \"• Use shorter generation/population counts for faster demos\"\n]\n\nfor tip in tips:\n    print(f\"   {tip}\")\n    \nprint()\nprint(\"🔗 Quick Start Template:\")\ntemplate = '''\n# === LockForge Quick Start Template ===\n\n# 1. Setup\n!git clone https://github.com/your-username/LockForge.git\n%cd LockForge\n\n# 2. Run AutoLock (fast settings for demo)\n!python AutoLock.py --in SRLL/SRLL_ISCAS_85/c499.bench --k 16 --pop 50 --gens 25 --seed 42\n\n# 3. Run CAS-Lock  \n!python CAS_Lock.py lock --input SRLL/SRLL_ISCAS_85/c499.bench --output c499_cas_demo.bench --N 6 --seed 42\n\n# 4. Validate\n!python CAS_Lock.py simulate --input c499_cas_demo.bench --key 101010101010 --orig SRLL/SRLL_ISCAS_85/c499.bench --random 500 --seed 1\n\n# 5. Analyze results\nimport json\nwith open('c499_autolock_key.txt', 'r') as f:\n    print(f\"Generated key: {f.read().strip()}\")\n'''\n\nprint(template)"

## 📚 Summary and Learning Objectives

### What We've Learned:

1. **Logic Locking Fundamentals**: Understanding the purpose and principles of hardware IP protection
2. **Multiple Techniques**: Explored AutoLock, CAS-Lock, DK-Lock, and overview of 7 additional methods
3. **Security Analysis**: How to validate locked circuits and measure corruption rates
4. **Practical Implementation**: Step-by-step guides for running each technique
5. **Comparison Framework**: Understanding trade-offs between security, complexity, and overhead

### Key Takeaways:

- **No single technique fits all scenarios** - choose based on specific requirements
- **Security validation is crucial** - always test with correct and wrong keys  
- **Area overhead vs security** - balance protection level with implementation cost
- **Temporal vs logical protection** - some techniques offer time-based security
- **Multi-layer approaches** - combining techniques can provide stronger protection

### Next Steps for Students:

1. **Hands-on Practice**: Run the techniques on different benchmark circuits
2. **Parameter Exploration**: Experiment with different key lengths and settings
3. **Attack Analysis**: Study how various attacks work and which techniques resist them
4. **Custom Implementations**: Try modifying techniques or creating hybrid approaches
5. **Research Current Trends**: Explore latest developments in hardware security

### Resources for Further Learning:

- **LockForge Documentation**: Individual README files for each technique
- **Academic Papers**: Research papers on logic locking and hardware security
- **Benchmark Suites**: ISCAS-85, ISCAS-89, ITC-99 for testing
- **Hardware Security Conferences**: HOST, CHES, DATE for latest research

---

**Course**: LLM4ChipDesign  
**Topic**: Hardware Logic Locking with LockForge  
**Objective**: Understand and apply IP protection techniques in chip design

*This notebook provides a comprehensive introduction to logic locking. For advanced topics and real-world deployment considerations, please refer to additional course materials and research literature.*